## Building AGentinc Workflows with LangChain

In [1]:
from typing import Annotated
import operator
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
 
class TicketState(TypedDict):
    customer_message: str
    log: Annotated[list, operator.add]
 
def log_received(state: TicketState) -> dict:
    return {"log": [f"Received: {state['customer_message']}"]}
 
def log_assigned(state: TicketState) -> dict:
    return {"log": ["Assigned to support queue"]}
 
builder = StateGraph(TicketState)
builder.add_node("log_received", log_received)
builder.add_node("log_assigned", log_assigned)
builder.add_edge(START, "log_received")
builder.add_edge("log_received", "log_assigned")
builder.add_edge("log_assigned", END)
graph = builder.compile()
 
result = graph.invoke({"customer_message": "My invoice looks wrong", "log": []})
print(result)

{'customer_message': 'My invoice looks wrong', 'log': ['Received: My invoice looks wrong', 'Assigned to support queue']}


# Calling the Model Inside a Node

In [2]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState # Ensure this is installed
from langchain_core.messages import SystemMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
 
def run_model(state: MessagesState) -> dict:
    system = SystemMessage("You are a support agent for a SaaS product. "
                           "Be concise and helpful.")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}



Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage
 
builder = StateGraph(MessagesState)
builder.add_node("run_model", run_model)
builder.add_edge(START, "run_model")
builder.add_edge("run_model", END)
 
graph = builder.compile()
 
result = graph.invoke({"messages": [HumanMessage("My dashboard isn't loading. What should I try?")]})
print(result["messages"][-1].content)

Please try these steps:

1.  **Refresh** the page.
2.  **Clear your browser's cache and cookies.**
3.  Try an **incognito/private window** or a different browser.
4.  Check our [Status Page](link to your status page) for any known issues.

If it still doesn't load, please let us know.
